## Baixando imagens de exames de raio-x torácico

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("khanfashee/nih-chest-x-ray-14-224x224-resized")

print("Path to dataset files:", path)

100%|██████████| 2.30G/2.30G [00:21<00:00, 112MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized/versions/3


In [2]:
images_per_class = 15

In [3]:
import os
import pandas as pd
import shutil
import kagglehub
from sklearn.utils import shuffle

# Baixar dataset
path = kagglehub.dataset_download("khanfashee/nih-chest-x-ray-14-224x224-resized")
print("Dataset path:", path)

csv_path = os.path.join(path, "Data_Entry_2017.csv")
image_folder = os.path.join(path, "images")

# Ler CSV
df = pd.read_csv(csv_path)

Dataset path: /root/.cache/kagglehub/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized/versions/3


##Conferindo

In [4]:
print(df)

             Image Index          Finding Labels  Follow-up #  Patient ID  \
0       00000001_000.png            Cardiomegaly            0           1   
1       00000001_001.png  Cardiomegaly|Emphysema            1           1   
2       00000001_002.png   Cardiomegaly|Effusion            2           1   
3       00000002_000.png              No Finding            0           2   
4       00000003_000.png                  Hernia            0           3   
...                  ...                     ...          ...         ...   
112115  00030801_001.png          Mass|Pneumonia            1       30801   
112116  00030802_000.png              No Finding            0       30802   
112117  00030803_000.png              No Finding            0       30803   
112118  00030804_000.png              No Finding            0       30804   
112119  00030805_000.png              No Finding            0       30805   

       Patient Age Patient Gender View Position  OriginalImage[Width  Heigh

## Analisando a quantidade de imagens por tipo de doença

In [13]:
class_counts = df_single["Finding Labels"].value_counts()
print(class_counts)

Finding Labels
Infiltration          9551
Atelectasis           4212
Effusion              3959
Nodule                2706
Pneumothorax          2199
Mass                  2138
Consolidation         1314
Pleural_Thickening    1127
Cardiomegaly          1094
Emphysema              895
Fibrosis               727
Edema                  634
Pneumonia              307
Hernia                 110
Name: count, dtype: int64


## Relizando uma redistribuição de imagens e salvando

In [14]:
import os
import pandas as pd
import shutil
from sklearn.utils import shuffle
import numpy as np

# ===== CONFIG =====
TOTAL_IMAGES = 200
image_folder = os.path.join(path, "images-224", "images-224")
output_dir = "/content/chest_xray_proportional"
os.makedirs(output_dir, exist_ok=True)

# ===== FILTRAR SINGLE LABEL =====
df["label_count"] = df["Finding Labels"].apply(lambda x: len(x.split("|")))
df_single = df[df["label_count"] == 1]

# Remover No Finding se existir
df_single = df_single[df_single["Finding Labels"] != "No Finding"]

# ===== CONTAGEM REAL =====
class_counts = df_single["Finding Labels"].value_counts()
total_available = class_counts.sum()

print("Distribuição original:")
print(class_counts)

# ===== CÁLCULO PROPORCIONAL =====
proportions = class_counts / total_available
images_per_class = (proportions * TOTAL_IMAGES)

# Arredondamento inteligente
images_floor = np.floor(images_per_class).astype(int)
remaining = TOTAL_IMAGES - images_floor.sum()

# Distribuir o restante para maiores frações decimais
decimal_part = (images_per_class - images_floor).sort_values(ascending=False)

for i in decimal_part.index[:remaining]:
    images_floor[i] += 1

print("\nDistribuição final (200 imagens):")
print(images_floor)
print("Total final:", images_floor.sum())

# ===== SELECIONAR IMAGENS =====
selected_images = []

for disease, n_images in images_floor.items():
    subset = df_single[df_single["Finding Labels"] == disease]
    subset = shuffle(subset, random_state=42).head(n_images)
    selected_images.append(subset)

df_selected = pd.concat(selected_images)

# ===== COPIAR =====
copied = 0

for _, row in df_selected.iterrows():
    src = os.path.join(image_folder, row["Image Index"])
    dst = os.path.join(output_dir, row["Image Index"])

    if os.path.exists(src):
        shutil.copy(src, dst)
        copied += 1

print("\nTotal imagens copiadas:", copied)

Distribuição original:
Finding Labels
Infiltration          9551
Atelectasis           4212
Effusion              3959
Nodule                2706
Pneumothorax          2199
Mass                  2138
Consolidation         1314
Pleural_Thickening    1127
Cardiomegaly          1094
Emphysema              895
Fibrosis               727
Edema                  634
Pneumonia              307
Hernia                 110
Name: count, dtype: int64

Distribuição final (200 imagens):
Finding Labels
Infiltration          62
Atelectasis           27
Effusion              26
Nodule                17
Pneumothorax          14
Mass                  14
Consolidation          8
Pleural_Thickening     7
Cardiomegaly           7
Emphysema              6
Fibrosis               5
Edema                  4
Pneumonia              2
Hernia                 1
Name: count, dtype: int64
Total final: 200

Total imagens copiadas: 200


##Salvando em ZIP

In [15]:
import zipfile

zip_path = "/content/chest_xray_proportional.zip"

with zipfile.ZipFile(zip_path, "w") as zipf:
    for file in os.listdir(output_dir):
        zipf.write(os.path.join(output_dir, file), file)

print("Arquivo zip criado:", zip_path)

Arquivo zip criado: /content/chest_xray_proportional.zip


##Download

In [16]:
from google.colab import files
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>